# 04 — Autoencoder Anomaly Detection

Train a dense autoencoder on control traders and score all traders.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
from src.models.autoencoder import run as run_ae
ae_scores = run_ae(save=True)

In [ ]:
# Score distribution
fig, ax = plt.subplots()
for label, mask, c in [("Leaked", ae_scores["is_leaked_market"]==1, "crimson"), ("Control", ae_scores["is_leaked_market"]==0, "steelblue")]:
    ax.hist(ae_scores.loc[mask, "anomaly_score_ae"], bins=50, alpha=0.6, label=label, color=c, density=True)
ax.set_xlabel("Reconstruction Error (MSE)")
ax.set_ylabel("Density")
ax.set_title("Autoencoder Anomaly Score Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("notebooks/fig_ae_scores.png", dpi=150)
plt.show()

In [ ]:
# Training loss history
import pickle
with open("data/processed/autoencoder_meta.pkl", "rb") as f:
    meta = pickle.load(f)
fig, ax = plt.subplots()
ax.plot(meta["val_history"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE")
ax.set_title("Autoencoder Training Curve")
plt.tight_layout()
plt.savefig("notebooks/fig_ae_training.png", dpi=150)
plt.show()

In [ ]:
# Quick Precision@K
from src.eval.metrics import precision_at_k
for k in [10, 20, 50, 100, 200]:
    p = precision_at_k(ae_scores, "anomaly_score_ae", "is_leaked_market", k)
    print(f"  Precision@{k:4d}: {p:.4f}")